In [ ]:
# Импорт библиотек
import requests
import pandas as pd
import time
import os
from datetime import datetime
from dotenv import load_dotenv

In [ ]:
# Параметры подключения к API
# Токен нигде не хранится в репозитории — берём его из переменных окружения (.env),
# см. .env.example в корне проекта.
load_dotenv()
TOKEN = os.getenv("VK_API_TOKEN")
if not TOKEN:
    raise RuntimeError("Задайте VK_API_TOKEN в файле .env (см. .env.example)")

GROUP_ID = "-44121040"  # id вашего сообщества
COUNT = 100
TOTAL_POSTS = 9000
API_VERSION = "5.131"
url_wall = "https://api.vk.com/method/wall.get"
url_video = "https://api.vk.com/method/video.get"
params = {
    "access_token": TOKEN,
    "v": API_VERSION,
    "owner_id": GROUP_ID,
    "count": 1,
}


In [ ]:
#Количество постов
response = requests.get(url_wall, params=params).json()
total_posts = response["response"]["count"]
print(f"Всего постов: {total_posts}")

#Количество видео
response = requests.get(url_video, params=params).json()
total_videos = response["response"]["count"]
print(f"Всего видео: {total_videos}")

In [ ]:
#Функция получения постов с пагинацией
def get_all_posts():
    all_posts = []
    for offset in range(0, TOTAL_POSTS, COUNT):
        params = {
            "access_token": TOKEN,
            "v": API_VERSION,
            "owner_id": GROUP_ID,
            "count": COUNT,
            "offset": offset
        }
        response = requests.get(url_wall, params=params).json()
        
        if "response" in response:
            all_posts.extend(response["response"]["items"])
        else:
            print("Ошибка:", response)
            break  # Останавливаем, если ошибка

    return all_posts

In [ ]:
#Функция обработки постов
def process_posts(posts):
    data = []
    for post in posts:
        post_id = post["id"]
        date = datetime.utcfromtimestamp(post["date"]).strftime('%Y-%m-%d %H:%M:%S')
        views = post.get("views", {}).get("count", 0)
        likes = post["likes"]["count"]
        comments = post["comments"]["count"]
        text = post["text"]

        data.append({
            "Post ID": post_id,
            "Дата публикации": date,
            "Просмотры": views,
            "Лайки": likes,
            "Комментарии": comments,
            "Текст поста": text
        })
    
    return data

In [ ]:
posts = get_all_posts()
post_data = process_posts(posts)

# Сохранение сырых данных
df = pd.DataFrame(post_data)
os.makedirs("../data", exist_ok=True)
df.to_excel("../data/raw_vk_posts.xlsx", index=False)
print("Данные сохранены в ../data/raw_vk_posts.xlsx")

In [ ]:
# Проверка
df = pd.read_excel("../data/raw_vk_posts.xlsx")
print(f"Размер: {df.shape}")
df.head(5)

In [ ]:
#Топ хештегов
import re

hashtags = {}

for post in posts:
    tags = re.findall(r"#(\w+)", post["text"])
    for tag in tags:
        if tag not in hashtags:
            hashtags[tag] = 0
        hashtags[tag] += 1

# ТОП-10 популярных хештегов
top_hashtags = sorted(hashtags.items(), key=lambda x: x[1], reverse=True)[:10]
print("ТОП-10 хештегов:")
for tag, count in top_hashtags:
    print(f"#{tag} — {count} упоминаний")
